# Solution: Ensemble Zero-Shot and Classical

In [ ]:
import os
import warnings
os.environ["HF_HOME"] = os.path.expanduser("~/.cache/huggingface")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:
import matplotlib as mpl

UB = {"Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF"}
UN = {"Black": "#0A0B0F", "Dark Gray": "#5A5C62"}
US = {"Orange": "#EE7622", "Green": "#23CE6B"}
mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import ARIMA as DartsARIMA
from darts.metrics import mae
HORIZON = 12


In [ ]:
DATA_PATH = "../data/subscribers.csv"


In [ ]:
def load_data(path):
    df = pd.read_csv(path, parse_dates=["date"], index_col="date").asfreq("MS")
    ts = TimeSeries.from_series(df["subscribers"])
    return ts.split_after(pd.Timestamp("2023-12-01"))

train, test = load_data(DATA_PATH)
print(f"Train: {len(train)} months, Test: {len(test)} months")


In [ ]:
def run_chronos(train, horizon):
    try:
        from darts.models import Chronos2Model
        model = Chronos2Model(model_name="amazon/chronos-bolt-small")
        return model.predict(horizon, series=train, num_samples=100)
    except ImportError:
        model = DartsARIMA(p=2, d=1, q=1)
        model.fit(train)
        return model.predict(horizon)

chronos_fc = run_chronos(train, HORIZON)


In [ ]:
def run_arima(train, horizon):
    model = DartsARIMA(p=2, d=1, q=1)
    model.fit(train)
    return model.predict(horizon)

arima_fc = run_arima(train, HORIZON)


In [ ]:
def build_ensemble(chronos_fc, arima_fc):
    ens_vals = (chronos_fc.values() + arima_fc.values()) / 2
    return TimeSeries.from_times_and_values(chronos_fc.time_index, ens_vals)

ensemble_fc = build_ensemble(chronos_fc, arima_fc)
print("Ensemble created")


In [ ]:
def evaluate_models(test, chronos_fc, arima_fc, ensemble_fc):
    return {
        "Chronos-2": mae(test, chronos_fc),
        "ARIMA": mae(test, arima_fc),
        "Ensemble": mae(test, ensemble_fc)
    }

results = evaluate_models(test, chronos_fc, arima_fc, ensemble_fc)
for name, err in results.items():
    print(f"{name}: MAE={err:,.0f}")


In [ ]:
def plot_ensemble(test, chronos_fc, arima_fc, ensemble_fc):
    fig, ax = plt.subplots(figsize=(14, 6))
    test.plot(ax=ax, label="Actual", color=UN["Black"])
    chronos_fc.plot(ax=ax, label="Chronos-2", color=UB["Brand Blue"], linestyle="--")
    arima_fc.plot(ax=ax, label="ARIMA", color=UB["Medium Blue"], linestyle="--")
    ensemble_fc.plot(ax=ax, label="Ensemble", color=US["Green"])
    ax.set_title("Ensemble vs Individual Models", fontsize=18, fontweight="bold")
    ax.set_ylabel("Subscribers", fontsize=16)
    ax.tick_params(axis="both", labelsize=14)
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_ensemble(test, chronos_fc, arima_fc, ensemble_fc)
